# Sequence to Sequence Learning with Attention (LSTM)

This notebook continues the previous **Seq2Seq LSTM** notebook and adds an **attention mechanism** while keeping the model LSTM-based.

The goal is to preserve the learning flow of the previous notebook:

1. prepare German → English translation data,
2. build an `Encoder`,
3. build a `Decoder`,
4. connect them with `Seq2Seq`,
5. train and evaluate the model,

but replace the **single fixed context bottleneck** with a **new context vector at every decoder time-step**.

> **Important:** this notebook intentionally uses **LSTM, not GRU**.  
> It follows the attention logic from the video you studied:
>
> - a **bidirectional LSTM encoder**,
> - all encoder hidden states are kept,
> - attention is computed from the **previous decoder hidden state** and all encoder states,
> - the attention-weighted context vector is concatenated with the **current target embedding**,
> - that concatenated vector becomes the input to the decoder LSTM.

The data preparation and training structure are adapted to match the style of the previous notebook. The attention-specific architecture is explained step by step so that the tensor shapes are visible while reading.

## 1. Why do we need Attention?

In the basic Seq2Seq model, the encoder reads the entire source sentence and the decoder mainly receives information through the encoder's final hidden and cell states.

Conceptually:

$$
x_1,x_2,\ldots,x_S
\rightarrow
\text{Encoder}
\rightarrow
(h_S,c_S)
\rightarrow
\text{Decoder}
$$

This forces a potentially long source sentence into a small fixed-size representation. For long sequences, this can become an **information bottleneck**.

Attention changes the idea.

Instead of asking the decoder to rely only on the final encoder state, we keep:

$$
H = \{h_1,h_2,\ldots,h_S\}
$$

where every $h_j$ stores information about a source position.

Then, at decoder time-step $t$, the model asks:

> **Which encoder states are most useful for predicting the next target word right now?**

This produces a different set of attention weights at each decoder step.

### The key distinction

These three objects are easy to confuse:

- **`encoder_states`**: the memory bank created once by the encoder.  
  For one source sentence, these vectors stay fixed during decoding.
- **`attention`**: a set of weights computed again at every decoder step.
- **`context_vector`**: a weighted sum of the fixed encoder states using the current attention weights.

So:

$$
\text{encoder states are fixed}
$$

but

$$
\alpha_1 \neq \alpha_2 \neq \alpha_3
$$

and therefore

$$
c_1 \neq c_2 \neq c_3
$$

because the decoder hidden state changes after every predicted word.

## 2. Attention used in this notebook

For decoder step $t$, let:

- $s_{t-1}$: previous decoder hidden state,
- $h_j$: encoder state at source position $j$.

We first compute an **energy / alignment score**:

$$
e_{t,j} = f(s_{t-1}, h_j)
$$

Here, $f$ is a small learnable neural network.

Then Softmax converts the energy values into attention weights:

$$
\alpha_{t,j}
=
\frac{\exp(e_{t,j})}
{\sum_k \exp(e_{t,k})}
$$

so that:

$$
\sum_j \alpha_{t,j} = 1
$$

Finally, the context vector is:

$$
c_t = \sum_j \alpha_{t,j} h_j
$$

This notebook then follows the video architecture:

$$
[c_t;\,d(y_t)]
\rightarrow
\text{Decoder LSTM}
\rightarrow
s_t
\rightarrow
\text{Linear}
\rightarrow
\hat{y}_{t+1}
$$

where $d(y_t)$ is the embedding of the current decoder input token.

> This is why the context vector is used **before** the decoder LSTM in this implementation.

## 3. Preparing Data

To make this notebook easy to compare with the previous one, we keep the same general German → English `Multi30k` data pipeline.

The code below uses the **legacy torchtext API**, matching the earlier notebook. This is useful for studying the architecture side by side, even though newer torchtext versions may require a different data-loading API.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchtext.legacy.datasets import Multi30k
from torchtext.legacy.data import Field, BucketIterator

import spacy
import numpy as np

import random
import math
import time

We set random seeds so that results are more reproducible.

In [ ]:
SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

Load the German and English spaCy tokenizers.

If the language models are not installed:

```bash
python -m spacy download de_core_news_sm
python -m spacy download en_core_web_sm
```

In [ ]:
spacy_de = spacy.load("de_core_news_sm")
spacy_en = spacy.load("en_core_web_sm")

We keep the same tokenization style as the previous notebook.

The German source sentence is reversed before being fed into the model, while English is kept in its original order.

In [ ]:
def tokenize_de(text):
    return [tok.text for tok in spacy_de.tokenizer(text)][::-1]


def tokenize_en(text):
    return [tok.text for tok in spacy_en.tokenizer(text)]

In [ ]:
SRC = Field(
    tokenize=tokenize_de,
    init_token="<sos>",
    eos_token="<eos>",
    lower=True,
)

TRG = Field(
    tokenize=tokenize_en,
    init_token="<sos>",
    eos_token="<eos>",
    lower=True,
)

In [ ]:
train_data, valid_data, test_data = Multi30k.splits(
    exts=(".de", ".en"),
    fields=(SRC, TRG),
)

print(f"Number of training examples: {len(train_data.examples)}")
print(f"Number of validation examples: {len(valid_data.examples)}")
print(f"Number of testing examples: {len(test_data.examples)}")

Build vocabularies from the training set only.

Tokens appearing fewer than two times are represented by `<unk>`.

In [ ]:
SRC.build_vocab(train_data, min_freq=2)
TRG.build_vocab(train_data, min_freq=2)

print(f"Unique tokens in source vocabulary: {len(SRC.vocab)}")
print(f"Unique tokens in target vocabulary: {len(TRG.vocab)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 128

train_iterator, valid_iterator, test_iterator = BucketIterator.splits(
    (train_data, valid_data, test_data),
    batch_size=BATCH_SIZE,
    device=device,
)

# 4. Building the Attention Seq2Seq Model

We still have the same three high-level components:

1. `Encoder`
2. `Decoder`
3. `Seq2Seq`

The important changes happen inside the encoder and decoder.

---

## 4.1 Encoder: change to a Bidirectional LSTM

The previous encoder only processed the sentence in one direction.

Now we use:

```python
bidirectional=True
```

So each source position receives:

- one forward hidden representation,
- one backward hidden representation.

If each direction has hidden size $H$, then every encoder state has size:

$$
2H
$$

Therefore:

```text
encoder_states = [src_len, batch_size, 2 * hidden_size]
```

These encoder states are the **memory bank** that attention will read from.

### Why do we still need `hidden` and `cell`?

The decoder itself is **not bidirectional**.

The encoder produces:

```text
forward final hidden  -> H
backward final hidden -> H
```

Concatenating them gives:

```text
2H
```

but the decoder expects hidden size:

```text
H
```

Therefore, this implementation uses learned linear layers:

$$
2H \rightarrow H
$$

for both the final hidden state and final cell state.

This is a bridge between the bidirectional encoder and the unidirectional decoder.

> This bridge is an implementation choice. The important attention idea is still the use of **all encoder states**, not this specific projection.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, dropout):
        super().__init__()

        self.hid_dim = hid_dim

        self.embedding = nn.Embedding(input_dim, emb_dim)

        # One layer, bidirectional LSTM.
        # We intentionally do not use the LSTM's internal dropout argument here,
        # because the video implementation uses a single recurrent layer.
        self.rnn = nn.LSTM(
            emb_dim,
            hid_dim,
            num_layers=1,
            bidirectional=True,
        )

        # 2H -> H bridge for the decoder's initial hidden and cell states.
        self.fc_hidden = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_cell = nn.Linear(hid_dim * 2, hid_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src_len, batch_size]

        embedded = self.dropout(self.embedding(src))

        # embedded = [src_len, batch_size, emb_dim]

        encoder_states, (hidden, cell) = self.rnn(embedded)

        # Because the encoder is bidirectional:
        #
        # encoder_states = [src_len, batch_size, 2 * hid_dim]
        #
        # hidden = [2, batch_size, hid_dim]
        # cell   = [2, batch_size, hid_dim]
        #
        # hidden[0:1] = final forward hidden state
        # hidden[1:2] = final backward hidden state

        hidden = self.fc_hidden(
            torch.cat((hidden[0:1], hidden[1:2]), dim=2)
        )

        cell = self.fc_cell(
            torch.cat((cell[0:1], cell[1:2]), dim=2)
        )

        # hidden = [1, batch_size, hid_dim]
        # cell   = [1, batch_size, hid_dim]

        return encoder_states, hidden, cell

### What is fixed during decoding?

After this line:

```python
encoder_states, hidden, cell = encoder(src)
```

the encoder has finished processing the source sentence.

For that source sentence:

```python
encoder_states
```

remain unchanged.

However:

```python
hidden
cell
```

will be updated by the decoder after each generated target token.

That changing decoder hidden state causes the **attention weights to change**, even though `encoder_states` themselves do not change.

## 4.2 Decoder with Attention

At a decoder time-step we have:

```text
input token
previous decoder hidden
previous decoder cell
all encoder states
```

The decoder performs five main operations:

1. Embed the current input token.
2. Compare the previous decoder hidden state with every encoder state.
3. Apply Softmax to obtain attention weights.
4. Compute an attention-weighted context vector.
5. Concatenate the context vector with the current embedding and feed them into the LSTM.

The output hidden state is then mapped to the target vocabulary using a linear layer.

### Why does the energy layer take `3 * hidden_size`?

For one source position:

```text
decoder hidden = H
encoder state  = 2H   # forward + backward
```

Concatenate them:

$$
H + 2H = 3H
$$

Therefore:

```python
nn.Linear(hid_dim * 3, 1)
```

takes the combined representation and produces one scalar energy value for each source position.

### Why do we repeat the decoder hidden state?

Suppose the source has $S$ tokens.

The encoder has:

```text
h1
h2
...
hS
```

but at decoder step $t$, we only have one previous decoder hidden state:

```text
s_(t-1)
```

To compare that same decoder state with every encoder state, we conceptually need:

```text
(s_(t-1), h1)
(s_(t-1), h2)
...
(s_(t-1), hS)
```

So we use:

```python
hidden.repeat(sequence_length, 1, 1)
```

This does **not** create new semantic information. It only matches tensor dimensions so that the same decoder state can be paired with every source position.

### Shape flow of Attention

Let:

- $S$ = source length,
- $B$ = batch size,
- $H$ = hidden size,
- $E$ = embedding size.

Before attention:

```text
encoder_states : [S, B, 2H]
hidden         : [1, B, H]
```

Repeat hidden:

```text
[S, B, H]
```

Concatenate:

```text
[S, B, H] + [S, B, 2H]
        -> [S, B, 3H]
```

Energy layer:

```text
Linear(3H -> 1)
        -> [S, B, 1]
```

Softmax across the source-length dimension:

```text
attention -> [S, B, 1]
```

For each example in the batch, the attention weights across the $S$ source positions sum to 1.

### Why `permute()` and `torch.bmm()`?

Mathematically, the context vector is simply:

$$
c_t = \sum_j \alpha_{t,j}h_j
$$

In code, we want to perform this weighted sum for every example in the batch.

After `permute`:

```text
attention      : [B, 1, S]
encoder_states : [B, S, 2H]
```

Then:

```python
torch.bmm(attention, encoder_states)
```

performs:

$$
[B,1,S] \times [B,S,2H]
\rightarrow
[B,1,2H]
$$

This is just the batched matrix-multiplication version of:

```text
alpha1*h1 + alpha2*h2 + ... + alphaS*hS
```

So `torch.bmm()` is not a new attention concept. It is only an efficient way to calculate the weighted sum.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, dropout):
        super().__init__()

        self.output_dim = output_dim
        self.hid_dim = hid_dim

        self.embedding = nn.Embedding(output_dim, emb_dim)

        # The LSTM receives:
        #   context vector = 2H
        #   current token embedding = E
        #
        # therefore input size = 2H + E.
        self.rnn = nn.LSTM(
            hid_dim * 2 + emb_dim,
            hid_dim,
            num_layers=1,
        )

        # Previous decoder hidden H + encoder state 2H = 3H.
        # Output one scalar energy score per source position.
        self.energy = nn.Linear(hid_dim * 3, 1)

        self.fc_out = nn.Linear(hid_dim, output_dim)

        # Normalize energy across the SOURCE SEQUENCE dimension.
        self.softmax = nn.Softmax(dim=0)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, encoder_states, hidden, cell):
        # input = [batch_size]
        # encoder_states = [src_len, batch_size, 2 * hid_dim]
        # hidden = [1, batch_size, hid_dim]
        # cell   = [1, batch_size, hid_dim]

        input = input.unsqueeze(0)

        # input = [1, batch_size]

        embedded = self.dropout(self.embedding(input))

        # embedded = [1, batch_size, emb_dim]

        sequence_length = encoder_states.shape[0]

        # Repeat the SAME previous decoder hidden state for every source position.
        h_reshaped = hidden.repeat(sequence_length, 1, 1)

        # h_reshaped = [src_len, batch_size, hid_dim]

        energy = self.relu(
            self.energy(
                torch.cat((h_reshaped, encoder_states), dim=2)
            )
        )

        # concatenated tensor = [src_len, batch_size, 3 * hid_dim]
        # energy              = [src_len, batch_size, 1]

        attention = self.softmax(energy)

        # attention = [src_len, batch_size, 1]
        #
        # For each item in the batch:
        # attention[0] + attention[1] + ... + attention[src_len-1] = 1

        # torch.bmm expects the batch dimension first.
        attention_bmm = attention.permute(1, 2, 0)

        # attention_bmm = [batch_size, 1, src_len]

        encoder_states_bmm = encoder_states.permute(1, 0, 2)

        # encoder_states_bmm = [batch_size, src_len, 2 * hid_dim]

        context_vector = torch.bmm(
            attention_bmm,
            encoder_states_bmm,
        )

        # context_vector = [batch_size, 1, 2 * hid_dim]

        context_vector = context_vector.permute(1, 0, 2)

        # context_vector = [1, batch_size, 2 * hid_dim]

        rnn_input = torch.cat(
            (context_vector, embedded),
            dim=2,
        )

        # rnn_input = [1, batch_size, 2 * hid_dim + emb_dim]

        output, (hidden, cell) = self.rnn(
            rnn_input,
            (hidden, cell),
        )

        # output = [1, batch_size, hid_dim]
        # hidden = [1, batch_size, hid_dim]
        # cell   = [1, batch_size, hid_dim]

        prediction = self.fc_out(output.squeeze(0))

        # prediction = [batch_size, output_dim]

        return prediction, hidden, cell

## 4.3 Decoder flow at one time-step

It is useful to mentally read the decoder in this order:

```text
previous decoder hidden s_(t-1)
           +
all fixed encoder states H
           |
           v
        ENERGY
           |
           v
        SOFTMAX
           |
           v
   attention weights alpha_t
           |
           v
 weighted sum of encoder states
           |
           v
      context c_t
           |
           +--------- current word embedding d(y_t)
           |                      |
           +---------- concat ----+
                       |
                       v
                 Decoder LSTM
                       |
                       v
                 new hidden s_t
                       |
                       v
                    Linear
                       |
                       v
              next-word prediction
```

The next decoder step uses the **new** hidden state $s_t$, so its attention distribution will usually be different.

## 4.4 Seq2Seq wrapper

The overall sequence-level logic is almost the same as before.

The major difference is:

```python
encoder_states, hidden, cell = self.encoder(src)
```

instead of only:

```python
hidden, cell = self.encoder(src)
```

Then the same `encoder_states` are passed to every decoder time-step.

Teacher forcing remains unchanged. Attention does **not** replace teacher forcing; the two mechanisms solve different problems.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

        assert encoder.hid_dim == decoder.hid_dim, (
            "Hidden dimensions of encoder and decoder must be equal!"
        )

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src = [src_len, batch_size]
        # trg = [trg_len, batch_size]

        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(
            trg_len,
            batch_size,
            trg_vocab_size,
        ).to(self.device)

        # Encoder runs ONCE.
        encoder_states, hidden, cell = self.encoder(src)

        # encoder_states stay fixed during the decoding loop.
        # hidden and cell will change after every decoder step.

        input = trg[0, :]  # <sos>

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(
                input,
                encoder_states,
                hidden,
                cell,
            )

            outputs[t] = output

            teacher_force = (
                random.random() < teacher_forcing_ratio
            )

            top1 = output.argmax(1)

            input = trg[t] if teacher_force else top1

        return outputs

# 5. Initializing the Model

The video implementation uses a **single recurrent layer**.

This matters because the bidirectional encoder already produces two directional states:

```text
forward  +  backward
```

Those are directions, not stacked LSTM layers.

So in this notebook we keep:

```text
1 encoder LSTM layer
1 decoder LSTM layer
```

and use dropout only around the embeddings, not as internal LSTM layer-to-layer dropout.

In [ ]:
INPUT_DIM = len(SRC.vocab)
OUTPUT_DIM = len(TRG.vocab)

ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512

ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

enc = Encoder(
    INPUT_DIM,
    ENC_EMB_DIM,
    HID_DIM,
    ENC_DROPOUT,
)

dec = Decoder(
    OUTPUT_DIM,
    DEC_EMB_DIM,
    HID_DIM,
    DEC_DROPOUT,
)

model = Seq2Seq(
    enc,
    dec,
    device,
).to(device)

model

Initialize weights in the same simple style as the previous notebook.

In [ ]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

In [ ]:
def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


print(
    f"The model has {count_parameters(model):,} "
    "trainable parameters"
)

We use Adam and cross-entropy loss.

Padding tokens are ignored when calculating the loss.

In [ ]:
optimizer = optim.Adam(model.parameters())

TRG_PAD_IDX = TRG.vocab.stoi[TRG.pad_token]

criterion = nn.CrossEntropyLoss(
    ignore_index=TRG_PAD_IDX
)

# 6. Training

The training loop is essentially the same as in the previous Seq2Seq notebook.

Attention is part of the model's forward computation, so PyTorch automatically backpropagates through:

```text
prediction
-> decoder LSTM
-> context vector
-> attention weights
-> energy network
-> encoder states
-> encoder LSTM
```

No separate "attention training loop" is required.

In [ ]:
def train(model, iterator, optimizer, criterion, clip):
    model.train()

    epoch_loss = 0

    for i, batch in enumerate(iterator):
        src = batch.src
        trg = batch.trg

        optimizer.zero_grad()

        output = model(src, trg)

        # output = [trg_len, batch_size, output_dim]
        # trg    = [trg_len, batch_size]

        output_dim = output.shape[-1]

        # Ignore the first <sos> position.
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)

        loss = criterion(output, trg)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            clip,
        )

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

Evaluation is the same except:

- gradients are disabled,
- parameters are not updated,
- teacher forcing is turned off.

In [ ]:
def evaluate(model, iterator, criterion):
    model.eval()

    epoch_loss = 0

    with torch.no_grad():
        for i, batch in enumerate(iterator):
            src = batch.src
            trg = batch.trg

            output = model(
                src,
                trg,
                teacher_forcing_ratio=0,
            )

            output_dim = output.shape[-1]

            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)

            loss = criterion(output, trg)

            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [ ]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time

    elapsed_mins = int(elapsed_time / 60)

    elapsed_secs = int(
        elapsed_time - elapsed_mins * 60
    )

    return elapsed_mins, elapsed_secs

Train the model and keep the weights with the best validation loss.

You can reduce `N_EPOCHS` while experimenting with the notebook.

In [ ]:
N_EPOCHS = 10
CLIP = 1

best_valid_loss = float("inf")

for epoch in range(N_EPOCHS):
    start_time = time.time()

    train_loss = train(
        model,
        train_iterator,
        optimizer,
        criterion,
        CLIP,
    )

    valid_loss = evaluate(
        model,
        valid_iterator,
        criterion,
    )

    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(
        start_time,
        end_time,
    )

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(
            model.state_dict(),
            "seq2seq-attention-lstm.pt",
        )

    print(
        f"Epoch: {epoch+1:02} | "
        f"Time: {epoch_mins}m {epoch_secs}s"
    )

    print(
        f"\tTrain Loss: {train_loss:.3f} | "
        f"Train PPL: {math.exp(train_loss):7.3f}"
    )

    print(
        f"\t Val. Loss: {valid_loss:.3f} | "
        f" Val. PPL: {math.exp(valid_loss):7.3f}"
    )

Load the best model and evaluate it on the test set.

In [ ]:
model.load_state_dict(
    torch.load(
        "seq2seq-attention-lstm.pt",
        map_location=device,
    )
)

test_loss = evaluate(
    model,
    test_iterator,
    criterion,
)

print(
    f"| Test Loss: {test_loss:.3f} | "
    f"Test PPL: {math.exp(test_loss):7.3f} |"
)

# 7. A Small Shape Walkthrough

Before worrying about a full dataset, it helps to imagine one decoder step with symbols.

Suppose:

```text
src_len = S
batch_size = B
hidden_size = H
embedding_size = E
```

The encoder gives:

```text
encoder_states : [S, B, 2H]
hidden         : [1, B, H]
cell           : [1, B, H]
```

Inside the decoder:

```text
hidden.repeat(S, 1, 1)
-> [S, B, H]

concat(hidden_repeated, encoder_states)
-> [S, B, 3H]

energy Linear(3H -> 1)
-> [S, B, 1]

Softmax over S
-> attention [S, B, 1]

permute attention
-> [B, 1, S]

permute encoder_states
-> [B, S, 2H]

bmm
-> [B, 1, 2H]

permute
-> context [1, B, 2H]

concat(context, embedding)
-> [1, B, 2H + E]

Decoder LSTM
-> output [1, B, H]

Linear(H -> target vocabulary)
-> prediction [B, output_dim]
```

If these shapes make sense, most of the implementation becomes mechanical rather than mysterious.

# 8. What Actually Changes From the Previous Seq2Seq Notebook?

The easiest way to study this notebook is to compare only these differences.

### Previous Encoder

Conceptually:

```python
outputs, (hidden, cell) = self.rnn(embedded)
return hidden, cell
```

### Attention Encoder

Now:

```python
encoder_states, (hidden, cell) = self.rnn(embedded)
return encoder_states, hidden, cell
```

We **keep all time-step states** because attention needs them.

---

### Previous Decoder input

Conceptually:

```python
embedded -> LSTM
```

### Attention Decoder input

Now:

```text
previous decoder hidden
        +
all encoder states
        |
        v
     attention
        |
        v
     context
        +
current embedding
        |
        v
       LSTM
```

---

### Previous Seq2Seq loop

Conceptually:

```python
hidden, cell = encoder(src)

for each target step:
    output, hidden, cell = decoder(
        input, hidden, cell
    )
```

### Attention Seq2Seq loop

Now:

```python
encoder_states, hidden, cell = encoder(src)

for each target step:
    output, hidden, cell = decoder(
        input,
        encoder_states,
        hidden,
        cell,
    )
```

`encoder_states` are reused unchanged.  
`hidden` and `cell` are replaced by their new decoder values every step.

# 9. Attention vs. the StatQuest Version You Noticed

You correctly noticed that not every Seq2Seq attention implementation places the context vector in the same location.

This notebook follows the video you are studying:

```text
previous hidden
    |
ATTENTION
    |
context + current embedding
    |
Decoder LSTM
    |
new hidden
    |
prediction
```

Another common presentation computes a decoder hidden state first and then combines:

```text
current decoder hidden + context
```

before the output projection.

Both belong to the broader family of encoder-decoder attention models, but their computation graphs are not identical.

For this notebook, you only need to remember the version used in the video:

$$
[s_{t-1}, H]
\rightarrow
\alpha_t
\rightarrow
c_t
$$

then:

$$
[c_t, d(y_t)]
\rightarrow
\text{LSTM}
\rightarrow
s_t
$$

then:

$$
s_t
\rightarrow
\text{Linear}
\rightarrow
\hat y_{t+1}
$$

# 10. Final Mental Model

If you remember only one picture from this notebook, remember this:

```text
SOURCE SENTENCE
      |
      v
Bidirectional LSTM Encoder
      |
      v
h1, h2, h3, ..., hS
      |
      |  fixed memory bank
      |
      +--------------------------------------+
                                             |
Decoder step t                               |
                                             |
previous hidden s_(t-1) --------------------+
          |
          v
compare with h1...hS
          |
          v
        energy
          |
          v
        softmax
          |
          v
 attention weights alpha_t
          |
          v
weighted sum of h1...hS
          |
          v
      context c_t
          |
          +------ embedding(current token)
                       |
                       v
                    concat
                       |
                       v
                 Decoder LSTM
                       |
                       v
                  new hidden s_t
                       |
                       v
                    Linear
                       |
                       v
                predicted word
```

The central idea is:

> **The encoder memory stays the same, but the decoder learns a different way to read that memory at every target time-step.**

That is the intuition behind attention in this lesson.

## 11. Video Result

In the video, the instructor reports that adding this attention mechanism improves the BLEU score from just under **21** in the previous model to approximately **27.5** in the attention model.

The exact score you obtain can differ because of:

- dataset/preprocessing details,
- random initialization,
- training settings,
- library versions,
- saved model parameters.

The important lesson is the architectural improvement, not reproducing the exact number.